# Reciprocal-rank fusion: a hands-on implementation

**Track:** Search · **Guide:** [`046-rrf.md`](../guides/concepts/046-rrf.md)

RRF combines ranked result lists using reciprocal rank, so systems with different score scales can contribute fairly.

> **Intuition:** A judge combines where each candidate placed on several independent leaderboards, not the raw points on each.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Reciprocal-rank fusion]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Blend BM25 and vector search for a mixed query containing both a part number and a natural-language description.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Create independent ranked lists

Lexical and vector systems produce ranks with incomparable score scales; RRF needs rank positions, not calibrated scores.


In [ ]:
lexical_ranking = ["part-E1042", "charger-guide", "battery-care"]
vector_ranking = ["battery-care", "charger-guide", "warranty"]

## 2. Implement reciprocal-rank fusion

Each input list contributes 1/(k + rank). Deduplicate within each list so duplicate hits cannot inflate a result.


In [ ]:
def reciprocal_rank_fusion(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, document_id in enumerate(dict.fromkeys(ranking), start=1):
            scores[document_id] = scores.get(document_id, 0.0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda item: (-item[1], item[0]))

fused = reciprocal_rank_fusion([lexical_ranking, vector_ranking])

## 3. Compare fused order

A result ranked highly by multiple methods is often promoted. Tune candidate depth and evaluate RRF against each retrieval baseline.


In [ ]:
for document_id, score in fused:
    print(f"{score:.4f} {document_id}")
assert fused[0][0] == "battery-care"

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/046-rrf.md) and the
[learning roadmap](../guides/ROADMAP.md).
